# Phase 1 — full-data retrain on T4

Trains all 5 architectures (iTransformer, LSTM, Bi-LSTM, 1D-CNN, time-axis Transformer) on the full PTB-XL @ 100 Hz, 3 seeds each, joint loss with α=0.05, β=2.0. ~60-90 min wall-clock on a T4. After the sweep, both zips (`runs/` and `checkpoints/`) get uploaded to a GitHub Release named `phase1-fullsweep`.

**Before running:**
1. Settings → Accelerator → **GPU T4 x1**
2. Settings → Internet → **On** (needed for `git clone`, W&B sync, GitHub release upload)
3. Add Input dataset: search **"PTB-XL ECG Dataset"** (`khyeh0719/ptb-xl-dataset`). The notebook auto-detects either Kaggle dataset path or falls back to a PhysioNet download.
4. Add-ons → Secrets:
   - `WANDB_API_KEY` — for live training curves at wandb.ai
   - `GH_TOKEN` — a GitHub personal access token with `repo` scope (Settings → Developer Settings → Tokens → Fine-grained → repo `DevDesai444/SmartECG`, contents=write). Without it the upload step is skipped and you can download the zips manually.
   - Toggle **Attach to notebook** on both.

Sequenced fastest-first so the headline iTransformer-vs-CNN comparison lands early; LSTM/Bi-LSTM are the long tail.

## env check

In [ ]:
import torch, sys, platform
print('python:', sys.version.split()[0])
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('platform:', platform.platform())

## clone the repo

In [ ]:
import os, subprocess
REPO = 'DevDesai444/SmartECG'  # also used by the release-upload step
REPO_URL = f'https://github.com/{REPO}.git'
WORK = '/kaggle/working/SmartECG'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, WORK], check=True)
%cd $WORK
!git log -1 --oneline

## point at PTB-XL

Try a few common Kaggle dataset paths; fall back to fetching from PhysioNet if none are attached. Either way the code expects `data/raw/ptbxl/ptbxl_database.csv` + `records100/`.

In [ ]:
from pathlib import Path
import shutil, subprocess

TARGET = Path('data/raw/ptbxl')
TARGET.mkdir(parents=True, exist_ok=True)

candidates = [
    '/kaggle/input/ptb-xl-ecg-dataset',
    '/kaggle/input/ptb-xl',
    '/kaggle/input/ptbxl',
]
found = None
for c in candidates:
    if (Path(c) / 'ptbxl_database.csv').exists() or list(Path(c).rglob('ptbxl_database.csv')):
        hits = list(Path(c).rglob('ptbxl_database.csv'))
        if hits:
            found = hits[0].parent
            break

if found:
    print(f'using attached dataset at {found}')
    for item in found.iterdir():
        link = TARGET / item.name
        if not link.exists():
            link.symlink_to(item)
else:
    print('no attached dataset — downloading PTB-XL from PhysioNet (~620 MB)')
    subprocess.run(['wget', '-q', '-O', str(TARGET / 'ptbxl_database.csv'),
        'https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv'], check=True)
    subprocess.run(['wget', '-q', '-O', str(TARGET / 'scp_statements.csv'),
        'https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv'], check=True)
    subprocess.run(['wget', '-q', '-r', '-np', '-nH', '--cut-dirs=3',
        '-P', str(TARGET), '-R', 'index.html*',
        'https://physionet.org/files/ptb-xl/1.0.3/records100/'], check=True)

print('records100 size:')
!du -sh data/raw/ptbxl/records100/ 2>/dev/null
!find data/raw/ptbxl/records100/ -name '*.dat' | wc -l

## deps — keep numpy<2 to avoid the pyarrow/sklearn break

In [ ]:
!pip install -q --upgrade 'numpy<2' 'shap<0.50' python-dotenv wfdb captum wandb onnx onnxruntime 2>&1 | tail -3
!pip install -q -e . 2>&1 | tail -3

## load secrets (wandb + github)

In [ ]:
import os
GH_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
        os.environ['WANDB_PROJECT'] = 'smartecg'
        os.environ['WANDB_GROUP'] = 'phase1-fullsweep'
        print('wandb: loaded; group=phase1-fullsweep')
    except Exception as e:
        print('wandb: no key — runs train locally only:', e)
        os.environ['WANDB_DISABLED'] = 'true'
    try:
        GH_TOKEN = secrets.get_secret('GH_TOKEN')
        print('github: token loaded — release upload will run at the end')
    except Exception as e:
        print('github: no token — final upload step will be skipped:', e)
except Exception as e:
    print('kaggle secrets unavailable:', e)
    os.environ['WANDB_DISABLED'] = 'true'

## sanity tests before kicking off the sweep

In [ ]:
!python -m pytest -q tests/ 2>&1 | tail -8

## the sweep

Sequenced fastest-first. Each architecture × 3 seeds. Output dirs are `runs/{model}/seed_{N}/` and `checkpoints/{model}/seed_{N}/` so nothing overwrites.

In [ ]:
import subprocess, time, json
from pathlib import Path

SEEDS = [42, 1337, 2024]
EPOCHS = 20  # early-stop on val macro AUROC, patience=5
MODELS = ['cnn1d', 'transformer_t', 'itransformer', 'lstm', 'bilstm']

summary = {}
for model in MODELS:
    summary[model] = []
    for seed in SEEDS:
        t0 = time.time()
        cmd = [
            'python', '-u', '-m', 'smartecg.training.train',
            '--config', f'configs/{model}.yaml',
            '--epochs', str(EPOCHS),
            '--seed', str(seed),
            '--tag', 'phase1',
        ]
        print(f'\n>>> {model}  seed={seed}', flush=True)
        rc = subprocess.run(cmd).returncode
        dt = time.time() - t0
        print(f'<<< exit={rc}  {dt:.0f}s', flush=True)
        summary[model].append({'seed': seed, 'exit': rc, 'wall_s': round(dt, 1)})

print('\n=== sweep done ===')
print(json.dumps(summary, indent=2))

## aggregate seeds.json per model

Loads the 3 `test_predictions.npz` per architecture, computes mean ± std on macro AUROC, F1, per-class AUROC, STEMI sens/spec, forecast MSE. One `runs/{model}/seeds.json` per architecture.

In [ ]:
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score

def per_class_metrics(y_true, y_score, classes, thr=0.5):
    y_pred = (y_score >= thr).astype(np.float32)
    out = {}
    aurocs = []; f1s = []
    for i, c in enumerate(classes):
        yt = y_true[:, i]; ys = y_score[:, i]; yp = y_pred[:, i]
        auroc = roc_auc_score(yt, ys) if len(set(yt)) == 2 else float('nan')
        f1 = f1_score(yt, yp, zero_division=0)
        tn = int(((yt == 0) & (yp == 0)).sum())
        fp = int(((yt == 0) & (yp == 1)).sum())
        tp = int(((yt == 1) & (yp == 1)).sum())
        fn = int(((yt == 1) & (yp == 0)).sum())
        sens = tp / (tp + fn) if (tp + fn) else float('nan')
        spec = tn / (tn + fp) if (tn + fp) else float('nan')
        out[c] = dict(auroc=auroc, f1=f1, sens=sens, spec=spec)
        aurocs.append(auroc); f1s.append(f1)
    out['macro'] = dict(auroc=float(np.nanmean(aurocs)), f1=float(np.nanmean(f1s)))
    return out

MODELS = ['cnn1d', 'transformer_t', 'itransformer', 'lstm', 'bilstm']
SEEDS = [42, 1337, 2024]

agg = {}
for model in MODELS:
    rows = []
    for seed in SEEDS:
        p = Path(f'runs/{model}/seed_{seed}/test_predictions.npz')
        if not p.exists():
            print(f'  missing {p} — skipping')
            continue
        z = np.load(p, allow_pickle=True)
        cls = list(z['classes'])
        m = per_class_metrics(z['y_true'], z['y_score'], cls)
        rows.append({'seed': int(seed),
                     'macro_auroc': m['macro']['auroc'],
                     'macro_f1': m['macro']['f1'],
                     'forecast_mse': float(z['test_forecast_mse']),
                     'per_class': {c: m[c] for c in cls}})
    if not rows:
        continue
    macro_aurocs = np.array([r['macro_auroc'] for r in rows])
    macro_f1s = np.array([r['macro_f1'] for r in rows])
    mse = np.array([r['forecast_mse'] for r in rows])
    summary = {
        'macro_auroc': {'mean': float(macro_aurocs.mean()), 'std': float(macro_aurocs.std(ddof=1))},
        'macro_f1':    {'mean': float(macro_f1s.mean()),    'std': float(macro_f1s.std(ddof=1))},
        'forecast_mse':{'mean': float(mse.mean()),          'std': float(mse.std(ddof=1))},
    }
    if rows:
        for c in rows[0]['per_class']:
            arr = np.array([r['per_class'][c]['auroc'] for r in rows])
            summary[f'auroc_{c}'] = {'mean': float(arr.mean()), 'std': float(arr.std(ddof=1))}
    out = {'model': model, 'seeds': rows, 'summary': summary}
    Path(f'runs/{model}/seeds.json').write_text(json.dumps(out, indent=2))
    agg[model] = summary
    print(f'{model:18s} AUROC = {summary["macro_auroc"]["mean"]:.4f} ± {summary["macro_auroc"]["std"]:.4f}')

print('\n=== per-architecture mean ± std macro AUROC ===')
for m, s in sorted(agg.items(), key=lambda kv: -kv[1]['macro_auroc']['mean']):
    print(f'  {m:18s} {s["macro_auroc"]["mean"]:.4f} ± {s["macro_auroc"]["std"]:.4f}')

## zip artifacts

In [ ]:
import shutil
RUNS_ZIP = '/kaggle/working/smartecg_phase1_artifacts.zip'
CKPT_ZIP = '/kaggle/working/smartecg_phase1_artifacts_ckpts.zip'
shutil.make_archive(RUNS_ZIP.replace('.zip', ''), 'zip', '.', 'runs')
shutil.make_archive(CKPT_ZIP.replace('.zip', ''), 'zip', '.', 'checkpoints')
!ls -lh /kaggle/working/smartecg_phase1_artifacts*

## push to a github release

Creates (or replaces) a release named `phase1-fullsweep` on the repo and uploads both zips as assets. The release notes get a short results summary so anyone landing on the release page can see the numbers without unzipping anything. Skipped silently if `GH_TOKEN` wasn't loaded.

In [ ]:
import os, json, requests, datetime
from pathlib import Path

if not GH_TOKEN:
    print('skipped — no GH_TOKEN. download the zips from /kaggle/working/ manually.')
else:
    TAG = 'phase1-fullsweep'
    REPO = 'DevDesai444/SmartECG'
    api = f'https://api.github.com/repos/{REPO}'
    H = {'Authorization': f'Bearer {GH_TOKEN}',
         'Accept': 'application/vnd.github+json',
         'X-GitHub-Api-Version': '2022-11-28'}

    # if a release with this tag exists, delete it (and its assets) so we can replace cleanly
    r = requests.get(f'{api}/releases/tags/{TAG}', headers=H)
    if r.status_code == 200:
        rel = r.json()
        print(f'  existing release id={rel["id"]} — deleting to replace')
        requests.delete(f'{api}/releases/{rel["id"]}', headers=H)
        # also delete the tag
        requests.delete(f'{api}/git/refs/tags/{TAG}', headers=H)

    # build a brief results table for the release notes
    lines = ['Full-data 3-seed sweep, joint loss α=0.05/β=2.0, PTB-XL @ 100Hz, fold 10 test.', '', '| model | macro AUROC (mean ± std) |', '|---|---|']
    for m, s in sorted(agg.items(), key=lambda kv: -kv[1]['macro_auroc']['mean']):
        lines.append(f'| {m} | {s["macro_auroc"]["mean"]:.4f} ± {s["macro_auroc"]["std"]:.4f} |')
    lines += ['', f'Generated {datetime.datetime.utcnow().isoformat(timespec="seconds")}Z on Kaggle T4.', '',
              'Assets:',
              '- `smartecg_phase1_artifacts.zip` — `runs/` (test_predictions.npz × 15, seeds.json × 5)',
              '- `smartecg_phase1_artifacts_ckpts.zip` — `checkpoints/` (best.pt × 15)']
    body = '\n'.join(lines)

    # create the release
    r = requests.post(f'{api}/releases', headers=H, data=json.dumps({
        'tag_name': TAG,
        'name': f'Phase 1 — full-data sweep',
        'body': body,
        'draft': False,
        'prerelease': False,
    }))
    r.raise_for_status()
    rel = r.json()
    upload_url = rel['upload_url'].split('{')[0]
    print(f'  release created: {rel["html_url"]}')

    # upload both zips as assets
    for zpath in [RUNS_ZIP, CKPT_ZIP]:
        name = Path(zpath).name
        sz = Path(zpath).stat().st_size
        print(f'  uploading {name} ({sz/1024/1024:.1f} MB)...')
        with open(zpath, 'rb') as f:
            ur = requests.post(
                f'{upload_url}?name={name}',
                headers={**H, 'Content-Type': 'application/zip'},
                data=f.read(),
            )
        ur.raise_for_status()
        print(f'  done: {ur.json()["browser_download_url"]}')

    print(f'\nrelease page: {rel["html_url"]}')

## what next

Once this finishes the GitHub release `phase1-fullsweep` will have both zips attached, plus a results summary in the release notes.

Locally:
1. `gh release download phase1-fullsweep -p '*.zip' -D /tmp/phase1` (or grab from the release page in a browser)
2. Unzip into the repo root: `unzip -o /tmp/phase1/smartecg_phase1_artifacts.zip` and the ckpts one too
3. Run `notebooks/01_explore_ptbxl.ipynb` for the full-data EDA pass
4. Update README results table from `runs/*/seeds.json`

Phase 2 (size ablation re-clean + 500 Hz) follows the same pattern — same notebook structure, swap the model list.